In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ====== CONFIGURATION ======
data_folder = "/content/drive/MyDrive/AIS_data"   # folder with monthly Hawaii_*.csv files
incident_file = "/content/hawaii_primary_trajectories_of_interest_2017_2020.csv"
# ===========================

In [ ]:
incidents_df = pd.read_csv('/content/hawaii_primary_trajectories_of_interest_2017_2020.csv')

# Convert datetime columns in incidents (as you already did)
incidents_df['ais_incident_start_bound_hst'] = pd.to_datetime(
    incidents_df['ais_incident_start_bound_hst']
)
incidents_df['ais_incident_end_bound_hst'] = pd.to_datetime(
    incidents_df['ais_incident_end_bound_hst']
)

In [ ]:
from geopy.distance import distance

def haversine_distance(lat1, lon1, lat2, lon2):
    """Return distance in kilometers between two lat/lon points."""
    return distance((lat1, lon1), (lat2, lon2)).km

In [ ]:
def process_monthly_ais(file_path, incidents_df):
    """
    Load, clean, add motion features, and label incidents for a single monthly AIS CSV.
    Returns a processed DataFrame.
    """
    # Load
    df = pd.read_csv(file_path)

    # Convert datetime columns
    df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
    df['datetime_hst'] = pd.to_datetime(df['datetime_hst'])

    # Keep only relevant columns
    keep_cols = [
        'MMSI', 'datetime_utc', 'datetime_hst', 'lat', 'lon',
        'speed_over_ground_knots', 'course_over_ground_deg',
        'vessel_type_code', 'length_m', 'width_m'
    ]
    df = df[keep_cols]

    # Fill missing length/width by vessel_type median
    df['length_m'] = df.groupby('vessel_type_code')['length_m'].transform(
        lambda x: x.fillna(x.median())
    )
    df['width_m'] = df.groupby('vessel_type_code')['width_m'].transform(
        lambda x: x.fillna(x.median())
    )
    # Fill missing vessel_type_code with mode
    df['vessel_type_code'] = df['vessel_type_code'].fillna(df['vessel_type_code'].mode()[0])
    # Drop rows missing SOG or COG
    df = df.dropna(subset=['speed_over_ground_knots', 'course_over_ground_deg'])

    # Sort by vessel and time
    df = df.sort_values(['MMSI', 'datetime_hst']).reset_index(drop=True)

    # Initialize motion columns
    df['delta_time_sec'] = np.nan
    df['delta_distance_km'] = np.nan
    df['computed_speed_knots'] = np.nan
    df['acceleration_knots_per_sec'] = np.nan
    df['heading_change_deg'] = np.nan

    for mmsi, group in df.groupby('MMSI'):
        idx = group.index
        if len(idx) < 2:
            continue
        prev_lat = group['lat'].shift(1)
        prev_lon = group['lon'].shift(1)
        prev_time = group['datetime_hst'].shift(1)
        prev_sog = group['speed_over_ground_knots'].shift(1)
        prev_cog = group['course_over_ground_deg'].shift(1)

        delta_sec = (group['datetime_hst'] - prev_time).dt.total_seconds()
        df.loc[idx, 'delta_time_sec'] = delta_sec

        # Distance using haversine
        dist_km = group.apply(
            lambda row: haversine_distance(prev_lat[row.name], prev_lon[row.name],
                                           row['lat'], row['lon'])
            if pd.notna(prev_lat[row.name]) else np.nan,
            axis=1
        )
        df.loc[idx, 'delta_distance_km'] = dist_km

        # Computed speed (knots)
        computed_knots = (dist_km / (delta_sec / 3600)) / 1.852
        df.loc[idx, 'computed_speed_knots'] = computed_knots

        # Acceleration
        accel = (group['speed_over_ground_knots'] - prev_sog) / delta_sec
        df.loc[idx, 'acceleration_knots_per_sec'] = accel

        # Heading change (smallest angle)
        cog_diff = np.abs(group['course_over_ground_deg'] - prev_cog)
        cog_diff = np.minimum(cog_diff, 360 - cog_diff)
        df.loc[idx, 'heading_change_deg'] = cog_diff

    # Drop first point of each vessel (NaN motion features)
    df = df.dropna(subset=['delta_time_sec']).reset_index(drop=True)

    # Add incident labels
    df['incident_num'] = np.nan
    df['ais_incident_type'] = np.nan
    df['ais_incident_evidence'] = np.nan

    for _, inc in incidents_df.iterrows():
        mmsi = inc['MMSI']
        start = inc['ais_incident_start_bound_hst']
        end = inc['ais_incident_end_bound_hst']
        mask = (df['MMSI'] == mmsi) & (df['datetime_hst'] >= start) & (df['datetime_hst'] <= end)
        df.loc[mask, 'incident_num'] = inc['incident_num']
        df.loc[mask, 'ais_incident_type'] = inc['ais_incident_type']
        df.loc[mask, 'ais_incident_evidence'] = inc['ais_incident_evidence']

    df['is_incident'] = df['incident_num'].notna().astype(int)

    return df

In [ ]:
import os
save_folder = "/content/drive/MyDrive/ais_processed_months/"
os.makedirs(save_folder, exist_ok=True)

In [ ]:
import glob
import os

data_folder = "/content/drive/MyDrive/AIS_data"   # raw CSV folder
all_raw_files = sorted(glob.glob(os.path.join(data_folder, "Hawaii_*.csv")))

processed_parquet_files = []

for raw_file in all_raw_files:
    # Derive output Parquet filename
    base_name = os.path.basename(raw_file).replace('.csv', '.parquet')
    out_path = os.path.join(save_folder, base_name)

    if os.path.exists(out_path):
        print(f"Skipping {base_name} – already processed")
        processed_parquet_files.append(out_path)
    else:
        print(f"Processing {os.path.basename(raw_file)}...")
        df_month = process_monthly_ais(raw_file, incidents_df)   # your function
        df_month.to_parquet(out_path, index=False)
        processed_parquet_files.append(out_path)
        print(f"  -> Saved {len(df_month)} rows to {out_path}")

Skipping Hawaii_2017_01.parquet – already processed
Skipping Hawaii_2017_02.parquet – already processed
Skipping Hawaii_2017_03.parquet – already processed
Skipping Hawaii_2017_04.parquet – already processed
Skipping Hawaii_2017_05.parquet – already processed
Skipping Hawaii_2017_06.parquet – already processed
Skipping Hawaii_2017_07.parquet – already processed
Skipping Hawaii_2017_08.parquet – already processed
Skipping Hawaii_2017_09.parquet – already processed
Skipping Hawaii_2017_10.parquet – already processed
Skipping Hawaii_2017_11.parquet – already processed
Skipping Hawaii_2017_12.parquet – already processed
Skipping Hawaii_2018_01.parquet – already processed
Skipping Hawaii_2018_02.parquet – already processed
Skipping Hawaii_2018_03.parquet – already processed
Skipping Hawaii_2018_04.parquet – already processed
Skipping Hawaii_2018_05.parquet – already processed
Skipping Hawaii_2018_06.parquet – already processed
Skipping Hawaii_2018_07.parquet – already processed
Skipping Haw

In [ ]:
import glob
import os

data_folder = "/content/drive/MyDrive/ais_processed_months"   # folder where monthly .parquet files are stored
output_folder = "/content/yearly_ais/"
os.makedirs(output_folder, exist_ok=True)

# Get all monthly parquet files
all_monthly = sorted(glob.glob(os.path.join(data_folder, "Hawaii_*.parquet")))

# Group by year
years = ['2017', '2018', '2019', '2020']
yearly_dfs = {}

for year in years:
    print(f"\nProcessing {year}...")

    # Check if yearly file already exists
    out_path = os.path.join(output_folder, f"hawaii_{year}.parquet")
    if os.path.exists(out_path):
        print(f"  {out_path} already exists, skipping...")
        continue

    year_files = [f for f in all_monthly if f"_{year}_" in f]
    if not year_files:
        print(f"  No files found for {year}")
        continue

    # Load each month and concatenate
    month_dfs = []
    for f in year_files:
        print(f"  Loading {os.path.basename(f)}")
        df = pd.read_parquet(f)
        month_dfs.append(df)

        # Combine months for this year
        yearly_df = pd.concat(month_dfs, ignore_index=True)

        # Filter bad records
        yearly_df = yearly_df[
            (yearly_df['computed_speed_knots'] >= 0) &
            (yearly_df['computed_speed_knots'] <= 60) &
            (yearly_df['acceleration_knots_per_sec'] >= -5) &
            (yearly_df['acceleration_knots_per_sec'] <= 5) &
            (yearly_df['heading_change_deg'] >= 0) &
            (yearly_df['heading_change_deg'] <= 180) &
            (yearly_df['delta_time_sec'] >= 1)
        ]

        print(f"Rows after filtering: {len(yearly_df)}")

    # Save yearly dataset
    out_path = os.path.join(output_folder, f"hawaii_{year}.parquet")
    yearly_df.to_parquet(out_path, index=False)
    print(f"  Saved to {out_path}")

    # Keep in dictionary if you need later
    yearly_dfs[year] = yearly_df

    # Free memory
    del month_dfs, yearly_df
    import gc; gc.collect()


Processing 2017...
  /content/yearly_ais/hawaii_2017.parquet already exists, skipping...

Processing 2018...
  /content/yearly_ais/hawaii_2018.parquet already exists, skipping...

Processing 2019...
  /content/yearly_ais/hawaii_2019.parquet already exists, skipping...

Processing 2020...
  /content/yearly_ais/hawaii_2020.parquet already exists, skipping...


In [ ]:
import os
import pandas as pd
import numpy as np
import gc

output_folder = "/content/yearly_ais/"
years = ['2017', '2018', '2019', '2020']

for year in years:
    file_path = os.path.join(output_folder, f"hawaii_{year}.parquet")
    if not os.path.exists(file_path):
        print(f"{file_path} not found, skipping.")
        continue

    print(f"Loading {file_path}...")
    yearly_df = pd.read_parquet(file_path)
    print(f"Original rows: {len(yearly_df)}")

    # Apply filtering
    yearly_df = yearly_df[
        (yearly_df['computed_speed_knots'] >= 0) &
        (yearly_df['computed_speed_knots'] <= 60) &
        (yearly_df['acceleration_knots_per_sec'] >= -5) &
        (yearly_df['acceleration_knots_per_sec'] <= 5) &
        (yearly_df['heading_change_deg'] >= 0) &
        (yearly_df['heading_change_deg'] <= 180) &
        (yearly_df['delta_time_sec'] >= 1)
    ]

    # Fix column types
    yearly_df['ais_incident_type'] = yearly_df['ais_incident_type'].astype('object')
    yearly_df['ais_incident_evidence'] = yearly_df['ais_incident_evidence'].astype('object')

    print(f"Rows after filtering: {len(yearly_df)}")

    # Save back (overwrite)
    yearly_df.to_parquet(file_path, index=False)
    print(f"Saved filtered version to {file_path}")

    del yearly_df
    gc.collect()

Loading /content/yearly_ais/hawaii_2017.parquet...
Original rows: 22216796
Rows after filtering: 22215608
Saved filtered version to /content/yearly_ais/hawaii_2017.parquet
Loading /content/yearly_ais/hawaii_2018.parquet...
Original rows: 21340964
Rows after filtering: 21333680
Saved filtered version to /content/yearly_ais/hawaii_2018.parquet
Loading /content/yearly_ais/hawaii_2019.parquet...
Original rows: 22677557
Rows after filtering: 22670306
Saved filtered version to /content/yearly_ais/hawaii_2019.parquet
Loading /content/yearly_ais/hawaii_2020.parquet...
Original rows: 22495521
Rows after filtering: 22476746
Saved filtered version to /content/yearly_ais/hawaii_2020.parquet


In [ ]:
import pandas as pd
import numpy as np
import os

# ==== CHANGE THIS PATH to one of your monthly parquet files ====
month_path = "/content/drive/MyDrive/ais_processed_months/Hawaii_2017_01.parquet"

# ===============================================================

print(f"Loading {os.path.basename(month_path)}...")
df = pd.read_parquet(month_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print("\nData types:")
print(df.dtypes)

# Check for nulls in critical columns
critical = ['MMSI', 'datetime_hst', 'lat', 'lon', 'computed_speed_knots',
            'acceleration_knots_per_sec', 'heading_change_deg', 'is_incident']
print("\nNull counts in critical columns:")
for col in critical:
    if col in df.columns:
        nulls = df[col].isnull().sum()
        print(f"  {col}: {nulls} ({nulls/len(df)*100:.2f}%)")

# Show a small sample
print("\nSample of 3 rows (first 10 columns):")
print(df.iloc[:3, :10])

# Show incident statistics
incident_rows = df['is_incident'].sum()
print(f"\nRows with incident label: {incident_rows} ({incident_rows/len(df)*100:.4f}%)")

# Free memory
del df

Loading Hawaii_2017_01.parquet...
Rows: 1,953,672
Columns: ['MMSI', 'datetime_utc', 'datetime_hst', 'lat', 'lon', 'speed_over_ground_knots', 'course_over_ground_deg', 'vessel_type_code', 'length_m', 'width_m', 'delta_time_sec', 'delta_distance_km', 'computed_speed_knots', 'acceleration_knots_per_sec', 'heading_change_deg', 'incident_num', 'ais_incident_type', 'ais_incident_evidence', 'is_incident']

Data types:
MMSI                                                           int64
datetime_utc                                     datetime64[ns, UTC]
datetime_hst                  datetime64[ns, pytz.FixedOffset(-600)]
lat                                                          float64
lon                                                          float64
speed_over_ground_knots                                      float64
course_over_ground_deg                                       float64
vessel_type_code                                             float64
length_m                         

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
import os

# Source folder (where your filtered yearly .parquet files are)
src_folder = "/content/yearly_ais/"
# Destination folder in Drive
dst_folder = "/content/drive/MyDrive/hawaii_ais_final/"
os.makedirs(dst_folder, exist_ok=True)

# Copy each file one by one (no pandas loading)
yearly_files = ["hawaii_2017.parquet", "hawaii_2018.parquet",
                "hawaii_2019.parquet", "hawaii_2020.parquet"]

for fname in yearly_files:
    src = os.path.join(src_folder, fname)
    dst = os.path.join(dst_folder, fname)
    if os.path.exists(src):
        print(f"Copying {fname}...")
        shutil.copy2(src, dst)
        size_gb = os.path.getsize(dst) / (1024**3)
        print(f"  -> Copied {size_gb:.2f} GB to Drive")
    else:
        print(f"Warning: {src} not found")

Copying hawaii_2017.parquet...
  -> Copied 0.79 GB to Drive
Copying hawaii_2018.parquet...
  -> Copied 0.76 GB to Drive
Copying hawaii_2019.parquet...
  -> Copied 0.79 GB to Drive
Copying hawaii_2020.parquet...
  -> Copied 0.77 GB to Drive
